# MobileSAM → CoreML

Exporta MobileSAM como **dos modelos CoreML separados**:
- `sam_encoder.mlpackage` — imagen 1024×1024 → embedding [1, 256, 64, 64] (corre cada 3-5 frames)
- `sam_decoder.mlpackage` — embedding + bbox YOLO → máscara [1, 1, 256, 256] (corre cada frame)

**⚠️ Antes de empezar:** Menú → Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ── 1. Dependencias ───────────────────────────────────────────────────────
!pip install git+https://github.com/ChaoningZhang/MobileSAM.git -q
!pip install coremltools onnx onnxruntime timm -q

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
# ── 2. Descargar pesos MobileSAM ──────────────────────────────────────────
import urllib.request, os

weights_url = 'https://github.com/ChaoningZhang/MobileSAM/raw/master/weights/mobile_sam.pt'
if not os.path.exists('mobile_sam.pt'):
    print('Descargando mobile_sam.pt...')
    urllib.request.urlretrieve(weights_url, 'mobile_sam.pt')

size_mb = os.path.getsize('mobile_sam.pt') / 1024 / 1024
print(f'✅ mobile_sam.pt — {size_mb:.1f} MB')

In [ ]:
# ── 3. Cargar modelo ──────────────────────────────────────────────────────
from mobile_sam import sam_model_registry

sam = sam_model_registry['vit_t'](checkpoint='mobile_sam.pt')
sam.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
sam = sam.to(device)
print(f'✅ MobileSAM cargado en {device}')

In [ ]:
# ── 4. Exportar ENCODER → ONNX ────────────────────────────────────────────
# Input:  [1, 3, 1024, 1024]  float32  (imagen BGR normalizada)
# Output: [1, 256, 64, 64]    float32  (image embedding)

import torch.nn as nn

class EncoderWrapper(nn.Module):
    def __init__(self, sam):
        super().__init__()
        self.encoder = sam.image_encoder
    def forward(self, x):          # x: [1, 3, 1024, 1024]
        return self.encoder(x)     # → [1, 256, 64, 64]

encoder = EncoderWrapper(sam).to(device).eval()
dummy_img = torch.zeros(1, 3, 1024, 1024, device=device)

torch.onnx.export(
    encoder, dummy_img,
    'sam_encoder.onnx',
    input_names  = ['image'],
    output_names = ['image_embeddings'],
    opset_version = 18,
    do_constant_folding = True,
)
print('✅ sam_encoder.onnx exportado')

In [ ]:
# ── 5. Exportar DECODER → ONNX ────────────────────────────────────────────
# Inputs:
#   image_embeddings: [1, 256, 64, 64]
#   point_coords:     [1, 2, 2]   (corners bbox: top-left, bottom-right en pixels 1024x1024)
#   point_labels:     [1, 2]      (2 = top-left corner, 3 = bottom-right corner)
# Output:
#   masks:            [1, 1, 256, 256]  (low-res mask, hay que upscale a 1024)
#   iou_predictions:  [1, 1]

class DecoderWrapper(nn.Module):
    def __init__(self, sam):
        super().__init__()
        self.sam = sam

    def forward(self, image_embeddings, point_coords, point_labels):
        sparse_emb, dense_emb = self.sam.prompt_encoder(
            points  = (point_coords, point_labels),
            boxes   = None,
            masks   = None,
        )
        low_res_masks, iou = self.sam.mask_decoder(
            image_embeddings         = image_embeddings,
            image_pe                 = self.sam.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings = sparse_emb,
            dense_prompt_embeddings  = dense_emb,
            multimask_output         = False,
        )
        return low_res_masks, iou   # [1,1,256,256], [1,1]

decoder = DecoderWrapper(sam).to(device).eval()

dummy_emb    = torch.zeros(1, 256, 64, 64, device=device)
dummy_coords = torch.zeros(1, 2, 2,       device=device)  # 2 puntos (bbox corners)
dummy_labels = torch.tensor([[2, 3]],     device=device, dtype=torch.float32)

torch.onnx.export(
    decoder,
    (dummy_emb, dummy_coords, dummy_labels),
    'sam_decoder.onnx',
    input_names  = ['image_embeddings', 'point_coords', 'point_labels'],
    output_names = ['masks', 'iou_predictions'],
    opset_version = 18,
    do_constant_folding = True,
)
print('✅ sam_decoder.onnx exportado')

In [ ]:
# ── 6. ONNX → CoreML ENCODER ─────────────────────────────────────────────
# coremltools necesita el objeto ONNX, no el path string
import onnx
import coremltools as ct

onnx_encoder = onnx.load('sam_encoder.onnx')

encoder_ct = ct.convert(
    onnx_encoder,
    inputs  = [ct.TensorType(name='image', shape=(1, 3, 1024, 1024))],
    outputs = [ct.TensorType(name='image_embeddings')],
    compute_units = ct.ComputeUnit.ALL,
    minimum_deployment_target = ct.target.iOS16,
)
encoder_ct.save('sam_encoder.mlpackage')
print('✅ sam_encoder.mlpackage guardado')

In [ ]:
# ── 7. ONNX → CoreML DECODER ─────────────────────────────────────────────
onnx_decoder = onnx.load('sam_decoder.onnx')

decoder_ct = ct.convert(
    onnx_decoder,
    inputs = [
        ct.TensorType(name='image_embeddings', shape=(1, 256, 64, 64)),
        ct.TensorType(name='point_coords',     shape=(1, 2, 2)),
        ct.TensorType(name='point_labels',     shape=(1, 2)),
    ],
    outputs = [
        ct.TensorType(name='masks'),
        ct.TensorType(name='iou_predictions'),
    ],
    compute_units = ct.ComputeUnit.ALL,
    minimum_deployment_target = ct.target.iOS16,
)
decoder_ct.save('sam_decoder.mlpackage')
print('✅ sam_decoder.mlpackage guardado')

In [ ]:
# ── 6b / 7b. FALLBACK: PyTorch → CoreML directo (sin ONNX) ───────────────
# Usar si la conversión ONNX falla. torch.jit.trace → ct.convert es más estable.

encoder_cpu = EncoderWrapper(sam).cpu().eval()
decoder_cpu = DecoderWrapper(sam).cpu().eval()

with torch.no_grad():
    traced_enc = torch.jit.trace(encoder_cpu, torch.zeros(1, 3, 1024, 1024))
    traced_dec = torch.jit.trace(
        decoder_cpu,
        (torch.zeros(1, 256, 64, 64),
         torch.zeros(1, 2, 2),
         torch.tensor([[2., 3.]]))
    )

enc_ct = ct.convert(
    traced_enc,
    inputs  = [ct.TensorType(name='image', shape=(1, 3, 1024, 1024))],
    outputs = [ct.TensorType(name='image_embeddings')],
    compute_units = ct.ComputeUnit.ALL,
    minimum_deployment_target = ct.target.iOS16,
)
enc_ct.save('sam_encoder.mlpackage')

dec_ct = ct.convert(
    traced_dec,
    inputs = [
        ct.TensorType(name='image_embeddings', shape=(1, 256, 64, 64)),
        ct.TensorType(name='point_coords',     shape=(1, 2, 2)),
        ct.TensorType(name='point_labels',     shape=(1, 2)),
    ],
    outputs = [
        ct.TensorType(name='masks'),
        ct.TensorType(name='iou_predictions'),
    ],
    compute_units = ct.ComputeUnit.ALL,
    minimum_deployment_target = ct.target.iOS16,
)
dec_ct.save('sam_decoder.mlpackage')
print('✅ Fallback PyTorch→CoreML: ambos modelos guardados')

In [ ]:
# ── 8. Verificar tamaños ──────────────────────────────────────────────────
import os

def dir_size_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / 1024 / 1024

print(f'sam_encoder.mlpackage : {dir_size_mb("sam_encoder.mlpackage"):.1f} MB')
print(f'sam_decoder.mlpackage : {dir_size_mb("sam_decoder.mlpackage"):.1f} MB')

In [ ]:
# ── 9. Comprimir y descargar ──────────────────────────────────────────────
import shutil
from google.colab import files

for name in ['sam_encoder', 'sam_decoder']:
    shutil.make_archive(f'/content/{name}.mlpackage', 'zip', f'/content/{name}.mlpackage')
    files.download(f'/content/{name}.mlpackage.zip')

print()
print('📋 Próximos pasos:')
print('1. Descomprimir sam_encoder.mlpackage.zip → sam_encoder.mlpackage/')
print('2. Descomprimir sam_decoder.mlpackage.zip → sam_decoder.mlpackage/')
print('3. Copiar ambas carpetas a modules/lidar-box-measure/ios/')
print('4. Actualizar LidarBoxMeasure.podspec:')
print("   s.resources = ['ios/box_detector.mlpackage',")
print("                  'ios/sam_encoder.mlpackage',")
print("                  'ios/sam_decoder.mlpackage']")

## Cómo usa Swift los dos modelos

```
ARFrame
  ↓ capturedImage (CVPixelBuffer landscape)
  ↓ resize → 1024×1024 → normalizar
sam_encoder.mlpackage
  ↓ image_embeddings [1, 256, 64, 64]   (cachear 3-5 frames)
  
YOLO detecta caja → bbox en coords de cámara
  ↓ convertir a espacio 1024×1024
sam_decoder.mlpackage
  inputs:
    image_embeddings [1, 256, 64, 64]
    point_coords     [1, 2, 2]   = [[x_tl, y_tl], [x_br, y_br]] en 1024×1024
    point_labels     [1, 2]      = [2, 3]  (2=top-left, 3=bottom-right)
  ↓ masks [1, 1, 256, 256]  → upscale bilinear → 1024×1024
  ↓ sigmoid > 0.5 → máscara binaria
  
LiDAR depth map
  ↓ filtrar solo pixels donde máscara == 1
  ↓ proyectar a 3D con intrínsecos de cámara
  ↓ AABB de puntos 3D
  ↓ largo / ancho / alto
```

### Coordenadas YOLO → SAM
YOLO da bbox normalizada [0,1] relativa al frame de cámara.
SAM encoder recibe imagen 1024×1024.
```swift
// YOLO: x1,y1,x2,y2 en [0,1] relativo a camera frame
// SAM:  multiplicar por 1024
let samX1 = yolo_x1 * 1024
let samY1 = yolo_y1 * 1024
let samX2 = yolo_x2 * 1024
let samY2 = yolo_y2 * 1024
```